In [1]:
import casadi as ca
import os

In [2]:
nx, ny = 3, 2

x_sym  = ca.SX.sym('x', nx)      # [n(1), v(1), a(1)]
dt_sym = ca.SX.sym('dt')         # scalar

# Unpack
q     = x_sym[0]               # joint position
dq     = x_sym[1]              # joint velocity
ddq     = x_sym[2]             # joint acceleration

# Discrete dynamics, constant acceleration over the step
q_next = q + dq*dt_sym + 0.5*ddq*(dt_sym**2)
dq_next = dq + ddq*dt_sym
ddq_next = ddq                        # zero jerk model

x_pred = ca.vertcat(q_next, dq_next, ddq_next)


# State Prediction and State Jacobian A = ∂f/∂x for EKF linearization
AF  = ca.jacobian(x_pred, x_sym)

In [3]:
# Define the measurement function.
# the measurements are: 
# [q_meas, dq_meas] 
h_expr = ca.vertcat(
    q,        # joint encoder position
    dq,        # joint velocity
)

# Compute the measurement Jacobian: H = ∂h/∂x
H_sym = ca.jacobian(h_expr, x_sym)

# a function for the measurement model and its Jacobian
h_func = ca.Function('h_func', [x_sym], [h_expr, H_sym])

In [4]:
# --- EKF iteration as a Casadi function ---
y_sym = ca.SX.sym('y', ny)       # measurement vector
P_sym = ca.SX.sym('P', nx, nx)   # error covariance

# Process and measurement noise covariances
Q_sym = ca.SX.sym('Q', nx, nx)   # process noise covariance
R_sym = ca.SX.sym('R', ny, ny)   # measurement noise covariance

# Prediction step:
#   x_pred = f(x,u,dt)
#   P_pred = AF * P * AF^T + Q
P_pred = AF @ P_sym @ AF.T + Q_sym

# Measurement update:
#   Compute the predicted measurement and its Jacobian
h_pred, H = h_func(x_pred)

# Innovation (measurement residual)
Y_corrected_k = ca.diag([1,  1])@y_sym #no scaling or correction applied


y_err = Y_corrected_k - h_pred
# Innovation covariance
S = H @ P_pred @ H.T + R_sym
# Kalman gain
K = P_pred @ H.T @ ca.inv(S)
# Updated state estimate
x_upd = x_pred + K @ y_err
# Updated covariance
P_upd = (ca.SX.eye(nx) - K @ H) @ P_pred

# Create the EKF update function.
ekf_update = ca.Function('ekf_update',
                           [x_sym, P_sym, dt_sym, y_sym, Q_sym, R_sym],
                           [x_upd, P_upd, y_err],
                           ['x', 'P','dt', 'y', 'Q', 'R'],
                           ['x_next', 'P_next', 'y_err'])

In [5]:
# ekf_update.save('EKF_update.casadi')

# c , cpp or matlab code generation for forward dynamics
ekf_update.generate("EKFnext.c")
os.system(f"gcc -fPIC -shared EKFnext.c -o libmEKF_next.so")

0

In [6]:
ekf_update(x=ca.DM.ones(nx), P=ca.DM.eye(nx), dt=0.1, y=ca.DM.zeros(ny), Q=ca.DM.eye(nx)*0.1, R=ca.DM.eye(ny)*0.1)

{'P_next': DM(
 [[0.0916783, 0.000691182, -0.000275097], 
  [0.000691182, 0.0916781, 0.00828731], 
  [-0.000275097, 0.00828731, 1.09173]]),
 'x_next': DM([0.0843518, 0.083903, 0.911879]),
 'y_err': DM([-1.105, -1.1])}

## Augmented arm disturbance EKF (simulator)

This follows the vehicle disturbance EKF architecture: nominal dynamics, analytic
CasADi Jacobians, random-walk disturbance, Joseph covariance update, generated C,
and a shared library called by the C++ hardware read loop.
The state is [q_e,q_d,q_c,q_b, dq_e,dq_d,dq_c,dq_b, d_e,d_d,d_c,d_b].
The additive disturbances are joint torques in Nm. Encoder q/dq are the eight
measurements. Bounded joint coordinates are not angle-wrapped.
The gripper retains the acceleration filter above.

Nominal prediction uses four RK4 elements for free motion and analytic locked
velocity decay, with the plant's known constraint equation. The C++ adapter
bounds integration chunks to at most 10 ms, propagates all elapsed time, and
corrects only once using the completed endpoint measurement. Locked disturbance means
are held and their process uncertainty grows; contact observations are censored rather than interpreted as external load.
Model mismatch remains part of the estimated load when observable. The nominal parameter
vector is independent of simulator plant parameter changes. The caller supplies
the previous applied actuator torque (after limits), the completed step's dt,
Q_d = Q_continuous * dt, and measurement covariance R. No truth disturbance or
future commanded torque is an estimator input. Covariances require tuning for
physical sensors; the defaults in the simulator are for encoder feedback.


In [ ]:
# Run this notebook from the usage directory.
import casadi as ca
import os
from pathlib import Path
import subprocess
import sys


def arm_disturbance_ekf_functions(project_root=None):
    if project_root is None:
        project_root = Path.cwd().parent
    project_root = Path(project_root).resolve()
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    from system.robot import RobotDynamics

    model = RobotDynamics(use_jit=False)
    model.from_file(str(project_root / 'usage/urdf/reach_alpha_5/alpha_5_robot.urdf'))
    # Build shared dynamics once; the caller bounds each integration interval.
    built = model.build_model('base_link', 'alpha_standard_jaws_base_link',
                      floating_base=True, has_endeffector=False,
                      step_discetization_no=1)
    args = model.F_next_reg_sys_arg
    # Keep the mass solve as an MX operation. Expanding a symbolic inverse
    # before AD duplicates large expressions and misses the control-loop budget.
    payload_torque = model.kinematic_dict['com_geo_J_q'][-1].T @ built[-1]
    passive = model.Cqdot_reg + model.id_g + model.B_reg + payload_torque
    terms = ca.Function('nominal_arm_terms', [args[0], args[3]],
                        [model.id_D, passive],
                        {'cse': True, 'is_diff_in': [True, False], 'jac_penalty': 0})
    z, torque, parameters, h = (ca.MX.sym('z', 8), ca.MX.sym('torque', 4),
                                ca.MX.sym('parameters', 81), ca.MX.sym('h'))
    mask = ca.MX.sym('lock_mask', 4)
    alpha = ca.MX.sym('baumgarte_alpha')
    mass, bias = terms(z, parameters)
    # Solve free rows of M a = tau-b with a_locked=-alpha*v_locked.
    # This is algebraically identical to the reaction-force solve, without
    # subtracting large nearly equal accelerations on locked axes.
    selector = ca.diag(mask)
    free = ca.MX.eye(4) - selector
    reduced_mass = free @ mass @ free + selector
    locked_acc = -alpha * selector @ z[4:]
    qdd = free @ ca.solve(reduced_mass, free @ (torque-bias-mass@locked_acc)) + locked_acc
    acceleration = ca.Function('nominal_arm_acceleration', [z, torque, parameters, mask, alpha], [qdd], {"is_diff_in": [True,True,False,False,False], "jac_penalty": 0})
    qdd = acceleration(z, torque, parameters, mask, alpha)
    ode = ca.Function('nominal_arm_ode', [z, torque, parameters, mask, alpha],
                      [ca.vertcat(z[4:], qdd)], {"is_diff_in": [True,True,False,False,False], "jac_penalty": 0})
    # Nominal parameters are fixed during state linearization. Reuse the sparse
    # full state Jacobian instead of dense parameter-direction derivatives.
    # Four RK4 elements approximate free motion per bounded interval.
    # Locked coordinates satisfy dv/dt=-alpha*v exactly. Use that closed
    # form at RK stage times; integrate only the coupled free coordinates.
    def locked_stage(candidate, t):
        decay=ca.exp(-alpha*t)
        travel=ca.if_else(alpha == 0, t, -ca.expm1(-alpha*t)/ca.if_else(alpha == 0, 1, alpha))
        exact=ca.vertcat(z[:4]+travel*z[4:], decay*z[4:])
        mask8=ca.vertcat(mask,mask)
        return (1-mask8)*candidate+mask8*exact
    k1 = ode(z, torque, parameters, mask, alpha)
    k2 = ode(locked_stage(z+h*k1/2,h/2), torque, parameters, mask, alpha)
    k3 = ode(locked_stage(z+h*k2/2,h/2), torque, parameters, mask, alpha)
    k4 = ode(locked_stage(z+h*k3,h), torque, parameters, mask, alpha)
    step = ca.Function('nominal_arm_step', [z, torque, h, parameters, mask, alpha],
                       [locked_stage(z+h*(k1+2*k2+2*k3+k4)/6,h)],
                       {"is_diff_in": [True,True,False,False,False,False], "jac_penalty": 0})
    x = ca.MX.sym('x', 12)
    u = ca.MX.sym('u_applied', 4)
    p = ca.MX.sym('p_nominal', 81)
    dt = ca.MX.sym('dt')
    qv = x[:8]
    for _ in range(4):
        qv = step(qv, u + x[8:], dt/4, p, mask, alpha)
    lower = ca.MX.sym('lower', 4)
    upper = ca.MX.sym('upper', 4)
    contact = ca.MX.sym('completed_contact', 4)
    # Match the plant's end-of-step unilateral position/velocity clamp.
    below, above = qv[:4] < lower, qv[:4] > upper
    projected_q = ca.fmin(upper, ca.fmax(lower, qv[:4]))
    projected_v = ca.if_else(below, ca.fmax(qv[4:], 0),
                             ca.if_else(above, ca.fmin(qv[4:], 0), qv[4:]))
    predicted = ca.vertcat(projected_q, projected_v, x[8:])
    # Completed-step contact is authoritative. Predicted contact alone must
    # not latch the observer at a stop after the measured joint releases.
    any_contact = ca.sum1(contact) > 0
    observable = (1-mask) * ca.if_else(any_contact, 0, 1)
    A = ca.jacobian(predicted, x)
    predict_limited = ca.Function('arm_disturbance_ekf_predict_limited',
        [x, u, p, dt, mask, alpha, lower, upper], [predicted, A])
    P = ca.MX.sym('P', 12, 12)
    Q = ca.MX.sym('Q_d', 12, 12)
    R = ca.MX.sym('R', 8, 8)
    y = ca.MX.sym('y', 8)
    # Hybrid mode-conditioned prediction: use the free-flow sensitivity
    # when measured contact clears, even if the stale nominal prediction
    # crosses a bound. Otherwise the projected Jacobian destroys the very
    # load sensitivity needed to learn the force releasing the joint.
    # Contact coordinates are censored observations of the free endpoint,
    # not evidence of zero external torque. Reset those state coordinates
    # to the completed measured clamp, with measurement covariance below.
    # Unclamped coordinates retain the coupled free-step sensitivities.
    contact8 = ca.vertcat(contact, contact)
    predicted = ca.vertcat(ca.if_else(contact8 > 0, y, qv), x[8:])
    A = ca.jacobian(predicted, x)
    H = ca.diag(1-contact8) @ ca.DM.eye(12)[:8, :]
    reset_noise = ca.diag(ca.vertcat(contact8 * ca.diag(R), ca.MX.zeros(4)))
    P_pred = A @ P @ A.T + Q + reset_noise
    innovation = (1-contact8) * (y - predicted[:8])
    S = H @ P_pred @ H.T + R
    K = ca.solve(S, H @ P_pred).T
    # Locked load is unobservable: retain its mean and process uncertainty.
    # Do not discard free-joint evidence during contact. The full correlated
    # covariance carries unresolved torque combinations; the separate
    # observable output remains conservative controller-use eligibility.
    K = ca.diag(ca.vertcat(ca.MX.ones(8), 1-mask)) @ K
    updated = predicted + K @ innovation
    residual = ca.MX.eye(12) - K @ H
    next_P = residual @ P_pred @ residual.T + K @ R @ K.T
    a = acceleration(updated[:8], u + updated[8:], p, mask, alpha)
    update_limited = ca.Function('arm_disturbance_ekf_update_limited',
        [x, P, u, p, dt, y, Q, R, mask, alpha, lower, upper, contact],
        [updated, (next_P + next_P.T)/2, innovation, S, a, observable],
        ['x', 'P', 'u_applied', 'p_nominal', 'dt', 'y', 'Q_d', 'R', 'lock_mask', 'alpha', 'lower', 'upper', 'completed_contact'],
        ['x_next', 'P_next', 'innovation', 'innovation_covariance', 'acceleration', 'disturbance_observable'])
    # Keep legacy entry points for unlocked/lock-only parity tests.
    lo, hi = ca.DM.ones(4)*-1e20, ca.DM.ones(4)*1e20
    predict_locked = ca.Function('arm_disturbance_ekf_predict_locked',
        [x, u, p, dt, mask, alpha], list(predict_limited(x,u,p,dt,mask,alpha,lo,hi)))
    update_locked = ca.Function('arm_disturbance_ekf_update_locked',
        [x,P,u,p,dt,y,Q,R,mask,alpha],
        list(update_limited(x,P,u,p,dt,y,Q,R,mask,alpha,lo,hi,ca.DM.zeros(4)))[:5])
    predict = ca.Function('arm_disturbance_ekf_predict', [x, u, p, dt],
        list(predict_locked(x, u, p, dt, ca.DM.zeros(4), 200.)))
    update = ca.Function('arm_disturbance_ekf_update', [x, P, u, p, dt, y, Q, R],
        list(update_locked(x, P, u, p, dt, y, Q, R, ca.DM.zeros(4), 200.)))
    # Same 30-element RK4 plant, with a shared reduced mass solve. Keep the
    # independent gripper spring/damper and the existing nine-input ABI.
    ps = ca.MX.sym('plant_state', 10)
    pu = ca.MX.sym('plant_input', 5)
    gm, gc, gk = (ca.MX.sym(name) for name in ('gripper_mass','gripper_damping','gripper_stiffness'))
    pz = ca.vertcat(ps[:4], ps[5:9])
    dz = ode(pz, pu[:4], p, mask, alpha)
    rhs = ca.vertcat(dz[:4], ps[9], dz[4:], (pu[4]-gc*ps[9]-gk*ps[4])/gm)
    plant_rhs = ca.Function('arm_plant_rhs',[ps,pu,p,gm,gc,gk,mask,alpha],[rhs])
    def rhs_at(state):
        return plant_rhs(state,pu,p,gm,gc,gk,mask,alpha)
    a1=rhs_at(ps)
    a2=rhs_at(ps+h*a1/2)
    a3=rhs_at(ps+h*a2/2)
    a4=rhs_at(ps+h*a3)
    plant_step=ca.Function('arm_plant_step',[ps,pu,h,p,gm,gc,gk,mask,alpha],
                          [ps+h*(a1+2*a2+2*a3+a4)/6])
    next_ps=ps
    for _ in range(30):
        next_ps=plant_step(next_ps,pu,dt/30,p,gm,gc,gk,mask,alpha)
    plant=ca.Function('arm_plant_next',[ps,pu,dt,p,gm,gc,gk,mask,alpha],[next_ps])
    return predict, update, predict_locked, update_locked, predict_limited, update_limited, plant


def export_arm_disturbance_ekf(output, project_root=None):
    output = Path(output).resolve()
    output.mkdir(parents=True, exist_ok=True)
    functions = arm_disturbance_ekf_functions(project_root)
    previous = Path.cwd()
    try:
        os.chdir(output)
        generator = ca.CodeGenerator('arm_disturbance_ekf.c', {'with_header': True})
        for function in functions:
            function.save(function.name() + '.casadi')
            generator.add(function)
        generator.generate()
        subprocess.run(['gcc', '-O2', '-fPIC', '-shared', 'arm_disturbance_ekf.c',
                        '-o', 'libmDisturbanceEKF_next.so', '-lm'], check=True)
    finally:
        os.chdir(previous)
    return functions

In [ ]:
# Set ARM_EKF_OUTPUT_DIR to choose the generated-artifact directory.
# Copy libmDisturbanceEKF_next.so to uvms-simulator/casadi_lib, then rebuild.
import os
from pathlib import Path
arm_ekf_output = Path(os.environ.get('ARM_EKF_OUTPUT_DIR', 'build/arm_disturbance_ekf'))
(arm_predict, arm_update, arm_predict_locked, arm_update_locked,
 arm_predict_limited, arm_update_limited, arm_plant) = export_arm_disturbance_ekf(arm_ekf_output)
